In [6]:
# import require library for preprocess
import mne
import numpy as np
from mne.channels import make_standard_montage
import matplotlib.pyplot as plt
from mne.datasets import eegbci
import scipy
import pickle 
import seaborn as sns

# import require library for classification
from sklearn.svm import SVC # SVM library
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis # LDA library
from sklearn.neighbors import KNeighborsClassifier # KNN library
from sklearn.model_selection import train_test_split, GridSearchCV, KFold

from sklearn.metrics import classification_report,confusion_matrix # Result representation
from os import listdir

# try to use ICA

In [7]:
subject = ["s008","s009", "s010", "s011", "s013","s014","s015", "s016", "s017", "s018","s019", "s021", "s022", "s023","s024", "s025", "s026", "s028"]
# subject = ["s008"]
condition = 'MI'

for subject in subject:
    fif_file_path = f"D:\\MTP_task\\MTP_eeg\\cleaned_data\\ICA_cleaned_Epoch_{subject}.fif"  # Replace with your actual .fif file path
    Epoch_ICA = mne.read_epochs(fif_file_path, preload=True, verbose=False)

    if condition == 'ME':
        # get first 240 epochs for ME
        Epochs = Epoch_ICA[:240]

    else:
        # get last 240 epochs for MI
        Epochs = Epoch_ICA[240:480]

    # Epochs.apply_baseline((-0.5, 0), verbose=False)  # Apply baseline correction
    Epochs.crop(tmin=-2.0, tmax=7.0, verbose=False)
    Epochs.drop_channels(['EXG1',
    'EXG2',
    'EXG3',
    'EXG4',
    'EXG5',
    'EXG6',
    'EXG7',
    'EXG8',
    'Status'])

    data = Epochs.get_data()  # shape: (n_trials, n_channels, n_samples)

    epochs_z = Epochs.copy()
    epochs_z._data = data

    # load pickle reject_index data
    if condition == 'ME':
        with open(f"D:\\MTP_task\\cam_data\\trajectory_label\\Reject_Labels_{subject}_ME.pkl", "rb") as f:
            reject_label_index = pickle.load(f)

        reject_number_1 = []
        reject_number_2 = []
        reject_number_3 = []
        reject_number_4 = []

        for i in reject_label_index:
            if i[0] == 'Number 1':
                reject_number_1.append(i[1])
            elif i[0] == 'Number 2':
                reject_number_2.append(i[1])
            elif i[0] == 'Number 3':
                reject_number_3.append(i[1])
            elif i[0] == 'Number 4':
                reject_number_4.append(i[1])
        # remove epoch trail based on reject_label_index


        # Option 1: create a new cleaned object
        epochs_z_clean_1 = epochs_z["Number 1"].copy().drop(reject_number_1)
        epochs_z_clean_2 = epochs_z["Number 2"].copy().drop(reject_number_2)
        epochs_z_clean_3 = epochs_z["Number 3"].copy().drop(reject_number_3)
        epochs_z_clean_4 = epochs_z["Number 4"].copy().drop(reject_number_4)

        epochs_z_clean = np.concatenate([epochs_z_clean_1.get_data(),
                                        epochs_z_clean_2.get_data(),
                                        epochs_z_clean_3.get_data(),
                                        epochs_z_clean_4.get_data()])

        labels_clean = np.concatenate([epochs_z_clean_1.events[:, 2],
                                    epochs_z_clean_2.events[:, 2],
                                    epochs_z_clean_3.events[:, 2],
                                    epochs_z_clean_4.events[:, 2]
                                    ])
    
    else:
        epochs_z_clean = epochs_z.get_data()
        labels_clean = epochs_z.events[:, 2]


    event_dict =  {'Number 1': 101,
        'Number 2': 102,
        'Number 3': 103,
        'Number 4': 104,
        }


    with open(f"D:\\MTP_task\\cam_data\\trajectory_label\\Trajectory_{subject}_{condition}.pkl", "rb") as f:
        class_velocities = pickle.load(f)


    # EEG_labels = Epochs.events[:, 2]  # Extract event labels from epochs
    EEG_labels = labels_clean  # Use cleaned labels
    code_to_stim = {v: k for k, v in event_dict.items()}

    # Prepare list to collect trials in EEG order
    velocity_stacked = {}
    velocity_stacked_new = {}


    for key in class_velocities.keys():
    # if key != 'IMU':
        print(f"{condition} Processing key: {key}")
        velocity_ordered = []
        for i, code in enumerate(EEG_labels):
            stim_name = code_to_stim[code]  # e.g., 105 -> 'UpRight'
            # Pop the first trial of this stimulus from velocity_label
            trial_data = class_velocities[key][stim_name][0]
            # Remove it from the stimulus array so the next same code uses the next trial
            class_velocities[key][stim_name] = np.delete(class_velocities[key][stim_name], 0, axis=0)
            # Append in EEG order
            velocity_ordered.append(trial_data[0:150,:])
        # Stack into final array: shape = (n_trials, nW_timepoints, 3)
        velocity_ordered = np.stack(velocity_ordered, axis=0)
        # Flatten time across epochs
        velocity_stacked[key] = velocity_ordered.reshape(-1, trial_data.shape[1])  # shape: (240*150, 3 or 6)
        velocity_stacked_new[key] = velocity_ordered  # shape: (n_trials, nW_timepoints, 3 or 6)

    np.save(f"D:\\MTP_task\\model\\trajectory_{subject}_{condition}.npy", velocity_stacked_new)
    np.save(f"D:\\MTP_task\\model\\EEG_data_{subject}_{condition}.npy", epochs_z_clean)
    np.save(f"D:\\MTP_task\\model\\labels_clean_{subject}_{condition}.npy", labels_clean)


C:\Users\user\AppData\Local\Temp\ipykernel_54840\2174330445.py:7: RuntimeWarning: This filename (D:\MTP_task\MTP_eeg\cleaned_data\ICA_cleaned_Epoch_s008.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  Epoch_ICA = mne.read_epochs(fif_file_path, preload=True, verbose=False)


MI Processing key: shoulder
MI Processing key: elbow
MI Processing key: wirst
MI Processing key: IMU


C:\Users\user\AppData\Local\Temp\ipykernel_54840\2174330445.py:7: RuntimeWarning: This filename (D:\MTP_task\MTP_eeg\cleaned_data\ICA_cleaned_Epoch_s009.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  Epoch_ICA = mne.read_epochs(fif_file_path, preload=True, verbose=False)


MI Processing key: shoulder
MI Processing key: elbow
MI Processing key: wirst
MI Processing key: IMU


C:\Users\user\AppData\Local\Temp\ipykernel_54840\2174330445.py:7: RuntimeWarning: This filename (D:\MTP_task\MTP_eeg\cleaned_data\ICA_cleaned_Epoch_s010.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  Epoch_ICA = mne.read_epochs(fif_file_path, preload=True, verbose=False)


...

MI Processing key: shoulder
MI Processing key: elbow
MI Processing key: wirst
MI Processing key: IMU


C:\Users\user\AppData\Local\Temp\ipykernel_54840\2174330445.py:7: RuntimeWarning: This filename (D:\MTP_task\MTP_eeg\cleaned_data\ICA_cleaned_Epoch_s026.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  Epoch_ICA = mne.read_epochs(fif_file_path, preload=True, verbose=False)


MI Processing key: shoulder
MI Processing key: elbow
MI Processing key: wirst
MI Processing key: IMU


C:\Users\user\AppData\Local\Temp\ipykernel_54840\2174330445.py:7: RuntimeWarning: This filename (D:\MTP_task\MTP_eeg\cleaned_data\ICA_cleaned_Epoch_s028.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  Epoch_ICA = mne.read_epochs(fif_file_path, preload=True, verbose=False)


MI Processing key: shoulder
MI Processing key: elbow
MI Processing key: wirst
MI Processing key: IMU


In [ ]:
velocity_stacked_new

{'shoulder': array([[[ 0.00862034,  0.00483737,  0.00102378],
         [ 0.00716425,  0.00407012,  0.00140068],
         [ 0.0045231 ,  0.0032091 ,  0.00138005],
         ...,
         [-0.00707824, -0.00781088, -0.01131915],
         [-0.00424488, -0.00459211, -0.00697598],
         [-0.00282644, -0.00340255, -0.00455882]],
 
        [[ 0.01007392,  0.00930262,  0.00280984],
         [ 0.00779056,  0.00688655,  0.00247448],
         [ 0.00408939,  0.00415642,  0.00139148],
         ...,
         [-0.0044724 , -0.00339379, -0.00782854],
         [-0.00240679, -0.00213001, -0.00435754],
         [-0.00214751, -0.0013267 , -0.00339311]],
 
        [[ 0.00632355,  0.00591697,  0.00124884],
         [ 0.00527482,  0.00526347,  0.00120972],
         [ 0.00305122,  0.00324468,  0.00113564],
         ...,
         [-0.00315761, -0.01369329, -0.0193943 ],
         [-0.00204925, -0.00836309, -0.01263098],
         [-0.00098335, -0.00646256, -0.0082315 ]],
 
        ...,
 
        [[ 0.00862034,

In [9]:
epochs_z_clean

array([[[ 3.87399359e-06,  3.99266673e-06,  4.38167399e-06, ...,
          4.27853156e-05,  4.36137270e-05,  4.45519363e-05],
        [ 8.40449957e-06,  8.72672899e-06,  9.44465137e-06, ...,
          4.26914230e-05,  4.40014192e-05,  4.54569708e-05],
        [ 7.85227621e-06,  7.77829973e-06,  7.76154684e-06, ...,
          5.20314279e-05,  5.30211546e-05,  5.39373214e-05],
        ...,
        [ 8.11309201e-06,  7.19637410e-06,  6.42588066e-06, ...,
         -8.50000652e-06, -7.38088374e-06, -6.21451591e-06],
        [ 5.90745458e-06,  4.60786578e-06,  3.42206522e-06, ...,
         -6.35166543e-07,  1.67922408e-07,  9.91796355e-07],
        [ 2.09802565e-05,  1.92163334e-05,  1.80188581e-05, ...,
         -2.87442231e-06, -9.08298375e-07,  1.23507414e-06]],

       [[ 4.67058235e-05,  4.61762829e-05,  4.51704436e-05, ...,
         -2.98287159e-06, -2.63543257e-06, -2.57101715e-06],
        [ 4.28035783e-05,  4.18253621e-05,  4.01923608e-05, ...,
          3.54914789e-07,  8.94954155e

In [10]:
labels_clean

array([103, 104, 102, 101, 104, 101, 101, 102, 102, 102, 101, 102, 101,
       103, 103, 103, 102, 104, 101, 103, 102, 104, 101, 102, 104, 103,
       101, 101, 103, 101, 103, 104, 103, 102, 104, 104, 103, 104, 104,
       102, 101, 103, 103, 104, 104, 102, 102, 103, 102, 104, 102, 101,
       101, 104, 101, 104, 101, 103, 103, 102, 102, 104, 101, 102, 102,
       103, 103, 101, 102, 101, 104, 102, 103, 104, 104, 104, 103, 101,
       101, 103, 102, 101, 101, 104, 102, 103, 102, 103, 102, 103, 104,
       103, 101, 103, 103, 101, 104, 103, 103, 102, 102, 102, 104, 104,
       102, 102, 101, 101, 104, 103, 103, 104, 101, 101, 101, 104, 104,
       104, 101, 102, 101, 101, 103, 102, 103, 104, 104, 103, 103, 101,
       104, 103, 104, 103, 102, 104, 104, 102, 102, 103, 102, 104, 102,
       101, 102, 102, 102, 101, 101, 103, 103, 101, 101, 101, 104, 104,
       102, 101, 103, 104, 101, 104, 104, 101, 103, 104, 102, 101, 102,
       102, 103, 104, 104, 102, 101, 104, 102, 102, 101, 104, 10